In [1]:
!pip install open_clip_torch huggingface_hub safetensors


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.2 MB/s eta 0:00:00


In [2]:
# 1. Cài đặt các thư viện cần thiết (nếu chưa có)


In [2]:
import torch
import open_clip
from huggingface_hub import hf_hub_download
from safetensors import safe_open
import os

model_id = 'timm/PE-Core-bigG-14-448'
print("Đã import xong các thư viện cần thiết!")

Đã import xong các thư viện cần thiết!


In [4]:
print("Đang tải file trọng số về thư mục chỉ định... (Vui lòng đợi)")

# Đường dẫn thư mục bạn muốn lưu (nằm ngay ngoài bộ nhớ phiên Colab)
target_dir = "/content/weights"

# Tải thẳng vào thư mục target_dir, bỏ qua cơ chế tạo cache ẩn phức tạp
weight_file_path = hf_hub_download(
    repo_id=model_id,
    filename="open_clip_model.safetensors",
    local_dir=target_dir
)

print(f"🎉 TẢI THÀNH CÔNG!")
print(f"-> File của bạn hiện đang nằm chính xác tại: {weight_file_path}")
print(f"-> Bạn có thể mở thư mục 'weights' ngay thanh bên trái để kiểm tra.")

Đang tải file trọng số về thư mục chỉ định... (Vui lòng đợi)


open_clip_model.safetensors:   0%|          | 0.00/9.68G [00:00<?, ?B/s]

🎉 TẢI THÀNH CÔNG!
-> File của bạn hiện đang nằm chính xác tại: /content/weights/open_clip_model.safetensors
-> Bạn có thể mở thư mục 'weights' ngay thanh bên trái để kiểm tra.


In [3]:
print("1. Đọc cấu hình chuẩn trực tiếp từ OpenCLIP...")
# Lấy file cấu hình nguyên bản từ Hugging Face (chỉ vài KB)
full_cfg = open_clip.get_model_config(f'hf-hub:{model_id}')

# Trích xuất cấu hình của riêng phần Text Encoder
if 'text_cfg' in full_cfg:
    text_cfg = full_cfg['text_cfg']
else:
    # Trường hợp cấu hình phẳng, lọc các tham số thuộc lớp CustomTextCLIP
    text_cfg = {k: v for k, v in full_cfg.items() if k in [
        'context_length', 'vocab_size', 'width', 'layers', 'heads', 'output_dim'
    ]}

print("\n📊 CẤU HÌNH TEXT ENCODER THỰC TẾ (Hãy kiểm tra tại đây):")
import json
print(json.dumps(text_cfg, indent=4))

1. Đọc cấu hình chuẩn trực tiếp từ OpenCLIP...



📊 CẤU HÌNH TEXT ENCODER THỰC TẾ (Hãy kiểm tra tại đây):
{
    "context_length": 72,
    "vocab_size": 49408,
    "width": 1280,
    "heads": 20,
    "layers": 24
}


In [4]:
import inspect
import open_clip

# Lấy danh sách các tham số mà hàm khởi tạo __init__ yêu cầu
init_signature = inspect.signature(open_clip.CustomTextCLIP.__init__)

print("🔍 CÁC THAM SỐ CHÍNH XÁC MÀ CustomTextCLIP YÊU CẦU:")
for name, param in init_signature.parameters.items():
    if name != 'self':
        print(f"  - {name}: {param.default if param.default != inspect.Parameter.empty else 'Bắt buộc phải truyền'}")

🔍 CÁC THAM SỐ CHÍNH XÁC MÀ CustomTextCLIP YÊU CẦU:
  - embed_dim: Bắt buộc phải truyền
  - vision_cfg: Bắt buộc phải truyền
  - text_cfg: Bắt buộc phải truyền
  - quick_gelu: False
  - init_logit_scale: 2.659260036932778
  - init_logit_bias: None
  - nonscalar_logit_scale: False
  - cast_dtype: None
  - output_dict: False


In [9]:
print("1. Khởi tạo kiến trúc Text Encoder thông qua cấu hình lai...")

# Lấy embed_dim gốc từ cấu hình (mặc định của bigG là 1280, nếu không thấy ta tự điền)
full_cfg = open_clip.get_model_config(f'hf-hub:{model_id}')
embed_dim = full_cfg.get("embed_dim", 1280)

# TẠO VISION CONFIG GIẢ LẬP để qua mặt hàm khởi tạo (Cực kỳ nhẹ, không tốn RAM)
fake_vision_cfg = {
    "layers": 1,
    "width": 8,
    "head_width": 8,
    "image_size": 224,
    "timm_model_name": None
}

# Gom toàn bộ vào bộ tham số chuẩn mà CustomTextCLIP yêu cầu
init_kwargs = {
    "embed_dim": embed_dim,
    "vision_cfg": fake_vision_cfg,
    "text_cfg": text_cfg,  # Sử dụng nguyên vẹn text_cfg chuẩn từ Cell 3
}

# Khởi tạo mô hình hỗn hợp (nhưng phần Vision gần như bằng 0)
full_fake_model = open_clip.CustomTextCLIP(**init_kwargs)

# Lấy riêng cục Text Encoder ra (đây mới là thứ ta cần)
text_model = full_fake_model.text
print("-> Đã dựng xong khung mô hình Text Encoder chuẩn xác từ file config!")


print("\n2. Tiến hành Lazy Load trích xuất trọng số từ ổ đĩa...")
text_state_dict = {}

# Đọc từng tensor từ file 10GB đã nằm trong thư mục /content/weights
with safe_open(weight_file_path, framework="pt", device="cpu") as f:
    for k in f.keys():
        # Lọc các trọng số thuộc về phần Text Encoder
        if k.startswith("text.") or k in ["token_embedding.weight", "positional_embedding"]:
            # Vì ta sẽ nạp vào full_fake_model (hoặc text_model),
            # Giữ nguyên key gốc có chữ "text." để nạp thông qua full_fake_model cho an toàn
            if k.startswith("text.") or k in ["token_embedding.weight", "positional_embedding"]:
                text_state_dict[k] = f.get_tensor(k)

print(f"-> Đã lọc xong! Số lượng tensors lấy ra: {len(text_state_dict)}")


print("\n3. Nạp trọng số và đóng gói file...")
# Nạp dữ liệu vào mô hình (cho phép strict=False vì mô hình có phần vision fake không có trọng số)
full_fake_model.load_state_dict(text_state_dict, strict=False)

# Giải phóng bộ nhớ đệm ngay để RAM luôn xanh
del text_state_dict

# Lưu thành file text_model.pt hoàn chỉnh của riêng phần Text (~1.3GB)
os.makedirs("my_pe_text_encoder", exist_ok=True)
output_path = "my_pe_text_encoder/text_model.pt"
torch.save(text_model.state_dict(), output_path)

print(f"🎉 THÀNH CÔNG RỰC RỠ! Đã ghi xong file hoàn chỉnh tại: {output_path}")

1. Khởi tạo kiến trúc Text Encoder thông qua cấu hình lai...
-> Đã dựng xong khung mô hình Text Encoder chuẩn xác từ file config!

2. Tiến hành Lazy Load trích xuất trọng số từ ổ đĩa...
-> Đã lọc xong! Số lượng tensors lấy ra: 293

3. Nạp trọng số và đóng gói file...
🎉 THÀNH CÔNG RỰC RỠ! Đã ghi xong file hoàn chỉnh tại: my_pe_text_encoder/text_model.pt


In [10]:
import torch
import open_clip

print("1. Đang nạp lại Text Encoder siêu nhẹ từ ổ đĩa...")

# Lấy embed_dim gốc từ cấu hình của bigG (mặc định là 1280)
full_cfg = open_clip.get_model_config(f'hf-hub:{model_id}')
embed_dim = full_cfg.get("embed_dim", 1280)

# Mạng Vision giả lập siêu tí hon để đáp ứng hàm khởi tạo (không tốn RAM)
fake_vision_cfg = {
    "layers": 1,
    "width": 8,
    "head_width": 8,
    "image_size": 224,
    "timm_model_name": None
}

init_kwargs = {
    "embed_dim": embed_dim,
    "vision_cfg": fake_vision_cfg,
    "text_cfg": text_cfg,  # Cấu hình chuẩn xịn thực tế từ Cell 3
}

# Khởi tạo mô hình cấu trúc lai
full_fake_model = open_clip.CustomTextCLIP(**init_kwargs)

# Nạp file trọng số sạch (~1.3GB) đã trích xuất vào riêng phần text của mô hình
full_fake_model.text.load_state_dict(torch.load("my_pe_text_encoder/text_model.pt", map_location="cpu"))

# Lấy riêng cục Text Encoder đã có trọng số ra để dùng cho Cell sau
model = full_fake_model.text
model.eval()  # Chuyển sang chế độ inference (đóng băng dropout)

# Khởi tạo sẵn bộ mã hóa từ vựng (Tokenizer) chuẩn của mô hình gốc
tokenizer = open_clip.get_tokenizer(f'hf-hub:{model_id}')

print("🎉 SẴN SÀNG! Mô hình Text và Tokenizer đã được nạp vào RAM thành công.")

1. Đang nạp lại Text Encoder siêu nhẹ từ ổ đĩa...
🎉 SẴN SÀNG! Mô hình Text và Tokenizer đã được nạp vào RAM thành công.


In [11]:
import numpy as np
import torch

# 💡 Thay đổi câu prompt bạn muốn test tại đây
query_text = '2 persons are in front of fruit stall in super market, with banner "Bán lẻ khai thác thị trường cho mục tiêu tăng trưởng 2 con số" and the big 23'

print(f"📝 Câu truy vấn của bạn:\n'{query_text}'")

# --- BƯỚC 1: XỬ LÝ VÀ PHÂN TÍCH TOKEN ---
# Tokenizer chuyển văn bản thành tensor có kích thước [1, context_length] (ở đây context_length = 72)
tokens = tokenizer([query_text])

# Đếm số token thực tế bằng cách tìm vị trí của token kết thúc chuỗi (End-of-Text token)
# Trong CLIP, các token trống ở cuối sẽ được lấp đầy bằng số 0 hoặc token ẩn.
token_list = tokens[0].tolist()
# Tìm độ dài thực tế trước khi bị đệm (padding) bằng cách lọc các token có nghĩa
# (Hàm tokenizer của open_clip mặc định đệm các số giống nhau ở cuối)
actual_token_count = len([t for t in token_list if t != 0])

print(f"\n📊 THÔNG SỐ TOKEN:")
print(f"   - Số lượng token thực tế: {actual_token_count} / 72 (tối đa)")
print(f"   - 10 Token IDs đầu tiên:  {token_list[:10]}...")

# --- BƯỚC 2: TRÍCH XUẤT EMBEDDING ---
with torch.no_grad():
    # Trích xuất vector 1280 chiều từ mô hình Text siêu nhẹ
    query_embedding = model(tokens)

    # Chuẩn hóa về Unit Vector (L2 Normalization)
    query_embedding /= query_embedding.norm(dim=-1, keepdim=True)

    # Kiểm tra lại độ dài hình học của vector (phải = 1.0)
    v_norm = query_embedding.norm(dim=-1).item()

    # Chuyển thành dạng NumPy array để tính toán với file .npy
    query_embedding_np = query_embedding.cpu().numpy().flatten()

print(f"\n⚡ THÔNG SỐ VECTOR:")
print(f"   - Kích thước Vector đầu ra (Shape): {query_embedding_np.shape}")
print(f"   - Độ dài hình học (L2 Norm):        {v_norm:.1f} (Chuẩn hóa thành công)")
print(f"   - 5 giá trị số đầu tiên của Vector: {query_embedding_np[:5].tolist()}...")

# --- BƯỚC 3: NẠP IMAGE EMBEDDING VÀ SO SÁNH ---
npy_path = "/content/000632.npy"
try:
    image_embedding_np = np.load(npy_path).flatten()

    # Tính Cosine Similarity bằng Tích vô hướng (Dot Product)
    cosine_sim = np.dot(query_embedding_np, image_embedding_np)

    print(f"\n🎯 KẾT QUẢ SO SÁNH NGỮ NGHĨA (Văn bản vs Hình ảnh):")
    print(f"   Score: {cosine_sim:.4f}")

except FileNotFoundError:
    print(f"\n❌ LỖI: Không tìm thấy file '{npy_path}'. Bạn hãy kiểm tra lại tab Tệp bên trái nhé!")

📝 Câu truy vấn của bạn:
'2 persons are in front of fruit stall in super market, with banner "Bán lẻ khai thác thị trường cho mục tiêu tăng trưởng 2 con số" and the big 23'

📊 THÔNG SỐ TOKEN:
   - Số lượng token thực tế: 71 / 72 (tối đa)
   - 10 Token IDs đầu tiên:  [49406, 273, 14592, 631, 530, 2184, 539, 5190, 12058, 530]...

⚡ THÔNG SỐ VECTOR:
   - Kích thước Vector đầu ra (Shape): (1280,)
   - Độ dài hình học (L2 Norm):        1.0 (Chuẩn hóa thành công)
   - 5 giá trị số đầu tiên của Vector: [-0.011486130766570568, -0.03701576963067055, 0.019633010029792786, -0.044887885451316833, -0.019395172595977783]...

🎯 KẾT QUẢ SO SÁNH NGỮ NGHĨA (Văn bản vs Hình ảnh):
   Score: 0.2878


In [31]:
!pip install -q onnx onnxruntime onnxscript

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 722.0/722.0 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 12.9 MB/s eta 0:00:00


In [12]:
import torch
import numpy as np
import onnx
from onnxruntime.quantization import quantize_dynamic, QuantType

print("1. Xuất mô hình PyTorch sang định dạng ONNX gốc (FP32)...")
# Chuẩn bị một input mẫu (đúng shape context_length = 72) để định hình graph
dummy_input = torch.zeros((1, 72), dtype=torch.long)

# Đường dẫn file ONNX chưa nén
onnx_model_path = "my_pe_text_encoder/text_model.onnx"

# Xuất file (model là biến mô hình text từ Cell 5 của bạn)
torch.onnx.export(
    model,
    dummy_input,
    onnx_model_path,
    input_names=['input_tokens'],
    output_names=['text_embeddings'],
    dynamic_axes={'input_tokens': {0: 'batch_size'}, 'text_embeddings': {0: 'batch_size'}},
)
print(f"-> Đã xuất xong ONNX gốc tại: {onnx_model_path}")

1. Xuất mô hình PyTorch sang định dạng ONNX gốc (FP32)...


/tmp/ipykernel_13871/548389853.py:14: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `TextTransformer([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `TextTransformer([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
-> Đã xuất xong ONNX gốc tại: my_pe_text_encoder/text_model.onnx


In [14]:
import gc
import torch

# 1. Xóa bỏ tận gốc các biến mô hình PyTorch đang chiếm RAM
if 'model' in locals(): del model
if 'full_fake_model' in locals(): del full_fake_model
if 'text_state_dict' in locals(): del text_state_dict

# 2. Kích hoạt bộ thu gom rác của Python và giải phóng bộ nhớ đệm PyTorch
gc.collect()
torch.cuda.empty_cache() if torch.cuda.is_available() else None

print("🧹 Đã dọn rác thành công! Hãy nhìn sang thanh tài nguyên RAM bên phải để kiểm tra nhé.")

🧹 Đã dọn rác thành công! Hãy nhìn sang thanh tài nguyên RAM bên phải để kiểm tra nhé.


In [15]:
import os
from onnxruntime.quantization import quantize_dynamic, QuantType

# Đường dẫn file ONNX gốc đã được xuất thành công ở phiên trước
onnx_model_path = "my_pe_text_encoder/text_model.onnx"
quantized_model_path = "my_pe_text_encoder/text_model_int8.onnx"

print("⚡ Đang tiến hành lượng tử hóa mô hình sang INT8 từ file ổ đĩa...")

# Nén động (Dynamic Quantization)
quantize_dynamic(
    model_input=onnx_model_path,
    model_output=quantized_model_path,
    weight_type=QuantType.QUInt8
)

print(f"🎉 XỬ LÝ THÀNH CÔNG TRONG VÒNG AN TOÀN!")
print(f"   - File ONNX 8-bit siêu nhẹ: {os.path.getsize(quantized_model_path) / (1024**2):.2f} MB")

⚡ Đang tiến hành lượng tử hóa mô hình sang INT8 từ file ổ đĩa...


🎉 XỬ LÝ THÀNH CÔNG TRONG VÒNG AN TOÀN!
   - File ONNX 8-bit siêu nhẹ: 515.79 MB


In [19]:
import numpy as np
import open_clip
import onnxruntime as ort
import time

print("1. Khởi tạo phiên chạy ONNX Runtime siêu nhẹ trên CPU...")
# Nạp file ONNX 8-bit vừa nén thành công
onnx_path = "my_pe_text_encoder/text_model_int8.onnx"
ort_session = ort.InferenceSession(onnx_path)

print("2. Chuẩn bị Tokenizer và câu prompt truy vấn...")
# Khởi tạo lại bộ Tokenizer chuẩn từ model_id ban đầu
model_id = 'timm/PE-Core-bigG-14-448'
tokenizer = open_clip.get_tokenizer(f'hf-hub:{model_id}')

# Câu prompt gốc từ Cell 6 của bạn
query_text = '2 persons are in front of fruit stall in super market, with banner "Bán lẻ khai thác thị trường cho mục tiêu tăng trưởng 2 con số" and the big 23'
print(f"📝 Câu truy vấn: '{query_text}'")

# ONNX Runtime nhận đầu vào là ma trận NumPy (dạng int64/long)
tokens = tokenizer([query_text]).numpy()

print("\n3. Thực thi truy vấn Embedding bằng mô hình nén INT8...")
start_time = time.time()

# Truy vấn mô hình ONNX
ort_inputs = {'input_tokens': tokens}
ort_outputs = ort_session.run(None, ort_inputs)
onnx_embedding = ort_outputs[0]

# Chuẩn hóa Vector đầu ra (L2 Normalization bằng NumPy)
onnx_embedding /= np.linalg.norm(onnx_embedding, axis=-1, keepdims=True)
print(onnx_embedding)
query_embedding_np = onnx_embedding.flatten()

end_time = time.time()
print(f"⚡ Thời gian chạy trích xuất vector: {(end_time - start_time)*1000:.2f} ms")

# --- SO SÁNH VỚI IMAGE EMBEDDING ---
npy_path = "/content/000632.npy"
try:
    image_embedding_np = np.load(npy_path).flatten()

    # Tính Cosine Similarity bằng Tích vô hướng
    cosine_sim = np.dot(query_embedding_np, image_embedding_np)

    print(f"\n🎯 KẾT QUẢ SO SÁNH (MÔ HÌNH NÉN INT8):")
    print(f"   - Score mới: {cosine_sim:.4f}")
    print(f"   - Đối chiếu Score gốc: 0.2878")
    print(f"   - Độ lệch thực tế: {abs(cosine_sim - 0.2878):.4f}")

except FileNotFoundError:
    print(f"\n❌ LỖI: Không tìm thấy file '{npy_path}'. Bạn hãy kiểm tra xem file còn nằm ở tab bên trái không nhé!")

1. Khởi tạo phiên chạy ONNX Runtime siêu nhẹ trên CPU...
2. Chuẩn bị Tokenizer và câu prompt truy vấn...
📝 Câu truy vấn: '2 persons are in front of fruit stall in super market, with banner "Bán lẻ khai thác thị trường cho mục tiêu tăng trưởng 2 con số" and the big 23'

3. Thực thi truy vấn Embedding bằng mô hình nén INT8...
[[-0.01092283 -0.02374284  0.00584203 ...  0.01097594  0.01140671
  -0.026953  ]]
⚡ Thời gian chạy trích xuất vector: 1121.76 ms

🎯 KẾT QUẢ SO SÁNH (MÔ HÌNH NÉN INT8):
   - Score mới: 0.2946
   - Đối chiếu Score gốc: 0.2878
   - Độ lệch thực tế: 0.0068


In [17]:
from google.colab import files
files.download("my_pe_text_encoder/text_model_int8.onnx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
import numpy as np
import open_clip
import onnxruntime as ort

class LightTextEmbedding:
    def __init__(self, model_path="text_model_int8.onnx", model_id="timm/PE-Core-bigG-14-448"):
        print("🚀 Khởi tạo Text Encoder INT8 siêu nhẹ...")
        # Load mô hình ONNX Runtime độc lập
        self.ort_session = ort.InferenceSession(model_path)
        # Load bộ Tokenizer chuẩn để xử lý text
        self.tokenizer = open_clip.get_tokenizer(f'hf-hub:{model_id}')

    def embed_text(self, text: str):
        # 1. Chuyển văn bản thành ma trận số dạng NumPy array
        tokens = self.tokenizer([text]).numpy()

        # 2. Truy vấn qua mô hình ONNX
        ort_inputs = {'input_tokens': tokens}
        ort_outputs = self.ort_session.run(None, ort_inputs)

        # 3. Trích xuất vector đầu ra và chuẩn hóa L2
        embedding = ort_outputs[0]
        embedding /= np.linalg.norm(embedding, axis=-1, keepdims=True)

        return embedding.flatten()

# === CÁCH SỬ DỤNG TRONG APP CỦA BẠN ===
if __name__ == "__main__":
    # Khởi tạo một lần duy nhất khi chạy app
    engine = LightTextEmbedding(model_path="/content/my_pe_text_encoder/text_model_int8.onnx")

    # Mỗi lần có câu prompt mới, chỉ cần gọi hàm này (chạy mất vài mili giây)
    # sentence = "Một người phụ nữ đang đứng trước quầy trái cây siêu thị"
    vector = engine.embed_text(query_text)

    print(f"🎉 Đã trích xuất xong vector embedding!")
    print(f"   - Kích thước vector: {vector.shape}") # Sẽ ra (1280,)
    print(f"   - 3 số đầu tiên: {vector[:3]}")

🚀 Khởi tạo Text Encoder INT8 siêu nhẹ...
🎉 Đã trích xuất xong vector embedding!
   - Kích thước vector: (1280,)
   - 3 số đầu tiên: [-0.01092283 -0.02374284  0.00584203]
